# Rust 111: Errors in Depth — Practice

Companion exercises for `11_errors-in-depth_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is at least one exercise for each of its demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. The Error Enum: Variants, Display, Error](#1-the-error-enum-variants-display-error)
- [2. From Conversions: The Upward Hop](#2-from-conversions-the-upward-hop)
- [3. Walking the Chain: `source()`](#3-walking-the-chain-source)
- [4. `Box<dyn Error>`: The Catch-All](#4-boxdyn-error-the-catch-all)
- [5. Returning Result from main: The Process Contract](#5-returning-result-from-main-the-process-contract)
- [6. Panics vs Recoverable Errors](#6-panics-vs-recoverable-errors)
- [7. Backtraces in Errors: The Stable Surface](#7-backtraces-in-errors-the-stable-surface)
- [8. The Cost Shape of `Result`](#8-the-cost-shape-of-result)
- [9. Reading the Compiler: E-Codes](#9-reading-the-compiler-e-codes)
- [11. Review: Mixed Problems](#11-review-mixed-problems)

§1–§9 each have exercises for the demonstration cells of the example notebook (the Contents lists only the exercise-bearing sections; §10's decision table is exercised in §11 together with everything else). When a check fails, the fastest route back is §11 of the example notebook, which is the cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A `fn` or `enum` may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers.

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Three notes specific to this chapter

- **Given code is complete and stays.** Cells hand you error enums so the stub has something to implement. Do not rename given items — later cells reuse them deliberately (`FeedError` from §1 returns in §2, §3 and §4; `PipelineError` from §2 returns in §3, §4 and §11; the §11 stubs return the §1–§4 cast to you one last time). This is the one-namespace rule working *for* you.
- **E-codes are the syllabus.** Several exercises are calibrated so the wrong approach produces the exact diagnostic the example notebook catalogues in §9 — E0277 (`?` without a `From` bridge), E0119 (blanket `From`), E0308 (`main`-shaped tails), E0252 (duplicate imports). When you see one, you are one step from the lesson.
- **No files, no crates.** Everything here is std-only and in-process; there is no scratch directory to clean. Panicking cells (§6) print a `panicked at` line from the kernel's hook *before* the exercise's own output — that is the payload being delivered, not a failure.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines that serve a stub you have not solved yet;
- `never read`/`never used` — stub-induced dead code: the check blocks that would read a given field sit below an unsolved stub.

Each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those families, it is yours.

**See**: §0 and §9 of the example notebook.


### Exercise 0.1 — The smoke test

**Goal**: prove the kernel compiles your items and runs your code, before any real exercise.

**Objectives**

- implement `kernel_pulse` to return the chapter's number
- keep the signature unchanged

**See**: §0.1 of the example notebook.

**Hint**: the body is one tail expression; no semicolon.


In [ ]:
/// Return the chapter number — the smallest possible proof the kernel is alive.
///
/// TODO: replace `todo!()` with the chapter number.
fn kernel_pulse() -> u32 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(kernel_pulse(), 11);
println!("Exercise 0.1 passed");


### Exercise 1.1 — The error enum's three faces

**Goal**: build a domain error with the required derive and both required impls, then prove each face (Debug shape, Display headline) works.

**Given**: the `RiskError` enum — `LimitBreach { symbol, px }` and `Halted(&'static str)` — is fully declared; only the impls are missing.

**Objectives**

- implement `Display` so `LimitBreach` prints `limit breach on {symbol} at {px}` and `Halted` prints `halted: {reason}`
- read the two given pieces and be able to say why they look the way they do: the derive is non-negotiable (`Error` demands `Debug` — E0277 names the trait declaration if you remove it), and the `Error` impl's **empty body** is the whole answer, because this error has no cause to expose
- keep the enum declaration and the given impls unchanged

**See**: §1.1 of the example notebook.

**Hint**: the `Display` impl is one `match` over the two variants; `write!` returns `fmt::Result`, so each arm is a tail expression.


In [ ]:
use std::fmt::{self, Display};

#[derive(Debug)]
enum RiskError {
    LimitBreach { symbol: String, px: u32 },
    Halted(&'static str),
}
impl Display for RiskError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        // TODO: replace `todo!()` with the match that writes both headlines.
        todo!()
    }
}
// The empty body IS the answer: no cause to expose, and the default source()
// returns None. Remove the derive above to meet E0277 - the compiler names
// `pub trait Error: Debug + Display` as the source of the bound.
impl std::error::Error for RiskError {}

// --- check (do not modify) ---
let breach = RiskError::LimitBreach { symbol: "ES".to_string(), px: 4_125 };
assert_eq!(breach.to_string(), "limit breach on ES at 4125");
assert_eq!(RiskError::Halted("circuit breaker").to_string(), "halted: circuit breaker");
assert!(format!("{breach:?}").contains("LimitBreach"));
println!("Exercise 1.1 passed");


### Exercise 1.2 — A wrapper that knows its cause

**Goal**: add the second error type of the chapter's cast: a variant that holds a cause as a field, plus the `Error` impl that exposes it.

**Given**: `RiskError` from §1.1; the `DeskError` enum is declared with one variant `Feed { source: RiskError, desk: &'static str }`.

**Objectives**

- implement `Display` for `DeskError` printing `desk {desk}: {source}` — the wrapper's headline **uses** its cause's headline
- implement `Error` for `DeskError` with a `source()` returning the cause
- keep the enum declaration unchanged

**See**: §1.2 of the example notebook.

**Hint**: `Display` can delegate — `write!(f, "desk {desk}: {source}")` formats the cause through *its* `Display`; `source()` returns `Some(source)` matched out of the variant.


In [ ]:
// (std::error::Error is imported here for the rest of the notebook.)
use std::error::Error;
#[derive(Debug)]
enum DeskError {
    Feed { source: RiskError, desk: &'static str },
}
impl Display for DeskError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        // TODO: replace `todo!()` with the headline, cause's Display included.
        todo!()
    }
}
impl std::error::Error for DeskError {
    // TODO: replace `todo!()` with the source() override.
    fn source(&self) -> Option<&(dyn std::error::Error + 'static)> {
        todo!()
    }
}

// --- check (do not modify) ---
let err = DeskError::Feed {
    source: RiskError::LimitBreach { symbol: "NQ".to_string(), px: 15_000 },
    desk: "rates",
};
assert_eq!(err.to_string(), "desk rates: limit breach on NQ at 15000");
assert!(err.source().is_some());
println!("Exercise 1.2 passed");


### Exercise 2.1 — The bridge that `?` rides on

**Goal**: write the `From` impl that lets a leaf error flow into a wrapper at a `?`, then use it in a two-hop function.

**Given**: `FeedError` — `BadQuote { raw }`, `Timeout(u32)` — is fully declared with its `Display`; the wrapper `PipelineError` is declared with one variant `Stage { source: FeedError, stage: &'static str }`, its `Display`, and its `Error` impl with `source()` (§1.2's pattern, given this time).

**Objectives**

- implement `From<FeedError> for PipelineError` stamping `stage: "ingest"`
- implement `ingest` to return `Ok(parsed)` on success, propagating both leaf failures with `?`
- keep the signatures unchanged

**See**: §2.1 of the example notebook.

**Hint**: the `From` body is the variant literal with `stage` hard-coded; after it exists, the two `?`s in `ingest` compile unchanged — that is the whole trick.


In [ ]:
#[derive(Debug)]
enum FeedError {
    BadQuote { raw: String },
    Timeout(u32),
}
impl Display for FeedError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            FeedError::BadQuote { raw } => write!(f, "malformed quote: {raw:?}"),
            FeedError::Timeout(ms) => write!(f, "timeout after {ms} ms"),
        }
    }
}
// Given, 1.1's lesson applied: a leaf error with no cause - empty body.
impl std::error::Error for FeedError {}
#[derive(Debug)]
enum PipelineError {
    Stage { source: FeedError, stage: &'static str },
}
impl Display for PipelineError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            PipelineError::Stage { stage, .. } => write!(f, "pipeline failed at stage {stage}"),
        }
    }
}
impl std::error::Error for PipelineError {
    fn source(&self) -> Option<&(dyn std::error::Error + 'static)> {
        match self {
            PipelineError::Stage { source, .. } => Some(source),
        }
    }
}
impl From<FeedError> for PipelineError {
    /// The bridge: every `?` from a FeedError into a PipelineError rides on this.
    ///
    /// TODO: replace `todo!()` with the Stage variant, stage stamped "ingest".
    fn from(source: FeedError) -> Self {
        todo!()
    }
}

fn parse_qty(raw: &str) -> Result<u32, FeedError> {
    raw.parse::<u32>().map_err(|_| FeedError::BadQuote { raw: raw.to_string() })
}
fn handshake(attempts: u32) -> Result<(), FeedError> {
    if attempts < 2 { Err(FeedError::Timeout(250)) } else { Ok(()) }
}
/// Ingest one quote: parse, then pretend to handshake. Both leaves are
/// FeedError; the signature promises PipelineError. `?` bridges the gap.
///
/// TODO: replace `todo!()` with the body: parse_qty(raw)?, handshake(2)?, Ok(qty).
fn ingest(raw: &str) -> Result<u32, PipelineError> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(ingest("300").expect("300 parses"), 300);
let err = ingest("abc").unwrap_err();
assert!(matches!(err, PipelineError::Stage { stage: "ingest", .. }));
assert_eq!(err.to_string(), "pipeline failed at stage ingest");
println!("Exercise 2.1 passed");


### Exercise 2.2 — `map_err` when the stamps disagree

**Goal**: skip the blanket `From` and stamp each callsite individually, so two failures from the same leaf get different context.

**Given**: `FeedError` and `PipelineError` from §2.1; `normalize` must label its two hops differently.

**Objectives**

- implement `normalize` to convert each leaf error with `map_err`, stamping `"normalize"` for the parse and `"handshake"` for the handshake
- keep the signature unchanged; do **not** add a `From` impl
- confirm the diagnostic: with no `From` bridge, a bare `?` on either hop is E0277 — the example notebook's §2 catalogue entry

**See**: §2.2 of the example notebook.

**Hint**: `.map_err(|e| PipelineError::Stage { source: e, stage: "..." })?` — one closure per callsite.


In [ ]:
/// Two hops, two stamps: map_err converts where each callsite stands.
///
/// TODO: replace `todo!()` with the body: parse_qty(raw) with the "normalize"
/// stamp, then handshake(attempts) with the "handshake" stamp, each via map_err + ?.
fn normalize(raw: &str, attempts: u32) -> Result<u32, PipelineError> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(normalize("150", 2).expect("150 parses"), 150); // both hops succeed
assert!(matches!(normalize("x", 2).unwrap_err(), PipelineError::Stage { stage: "normalize", .. }));
// "7" parses, but attempts = 1 fails the given handshake (needs >= 2):
assert!(matches!(normalize("7", 1).unwrap_err(), PipelineError::Stage { stage: "handshake", .. }));
println!("Exercise 2.2 passed");


### Exercise 2.3 — `?` on Option, and the one mixed hop

**Goal**: use `?` to early-return `None` from an `Option`-returning function, then convert the other way with `ok_or`.

**Objectives**

- implement `last_reported` to return the last candle's price scaled by `k`, or `None` when the window is empty — with exactly one `?`
- implement `require_last` to turn `Option<f64>` into `Result<f64, &'static str>` using `ok_or` and `?`
- keep the signatures unchanged

**See**: §2.3 of the example notebook.

**Hint**: `candles.last()?` is the early return; `opt.ok_or("empty window")?` is the mixed hop — the `?` then propagates the `Err`.


In [ ]:
/// The last price, scaled. Empty window means no answer, not a failure.
///
/// TODO: replace `todo!()` with the body using `?` on Option.
fn last_reported(candles: &[f64], k: f64) -> Option<f64> {
    todo!()
}
/// The last price, as a hard requirement: absence becomes an error here.
///
/// TODO: replace `todo!()` with the body using ok_or plus `?`.
fn require_last(candles: &[f64]) -> Result<f64, &'static str> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(last_reported(&[100.0, 200.0], 2.0), Some(400.0));
assert_eq!(last_reported(&[], 2.0), None);
assert_eq!(require_last(&[100.0, 200.0]), Ok(200.0));
assert_eq!(require_last(&[]), Err("empty window"));
println!("Exercise 2.3 passed");


### Exercise 3.1 — Walk the chain

**Goal**: traverse a wrapped error with `source()` and collect the chain — the loop `eyre`'s `Caused by:` block runs for you.

**Given**: `PipelineError::Stage { source, .. }` and `FeedError` from §2.1 — the chain here is exactly two nodes, as in the example notebook.

**Objectives**

- implement `chain_of` to return the headline of the outermost error plus every cause's headline, outermost first
- the first hop must use the fully qualified call `std::error::Error::source(&err)`; subsequent hops use the trait-object call
- keep the signature unchanged

**See**: §3.1 of the example notebook.

**Hint**: seed the `Vec` with `format!("{err}")`, bind `let mut hop = std::error::Error::source(&err);` and `while let Some(e) = hop` — pushing `format!("{e}")` and rebinding `hop = e.source()` inside.


In [ ]:
/// Headlines of the whole chain, outermost first.
///
/// TODO: replace `todo!()` with the chain walk.
fn chain_of(err: &dyn std::error::Error) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let err = PipelineError::Stage {
    source: FeedError::BadQuote { raw: "12,x5".to_string() },
    stage: "ingest",
};
let chain = chain_of(&err);
assert_eq!(chain.len(), 2);
assert_eq!(chain[0], "pipeline failed at stage ingest");
assert_eq!(chain[1], "malformed quote: \"12,x5\"");
println!("Exercise 3.1 passed");


### Exercise 3.2 — The typed accessor

**Goal**: give `PipelineError` the payload path that `source()` cannot offer: a typed view of its cause, plus the layer's own context.

**Objectives**

- implement `cause` returning `&FeedError` — the typed accessor, one `match` arm
- implement `stage` returning the wrapper's `&'static str`
- keep the signatures unchanged

**See**: §3.2 of the example notebook.

**Hint**: both are `match self { PipelineError::Stage { x, .. } => x }`-shaped; the difference is only which field comes out.


In [ ]:
impl PipelineError {
    /// Typed view of the cause: no dyn, no downcast, full exhaustiveness.
    ///
    /// TODO: replace `todo!()` with the match returning the cause.
    fn cause(&self) -> &FeedError {
        todo!()
    }
    /// The layer's own context, readable without touching the cause.
    ///
    /// TODO: replace `todo!()` with the match returning the stage.
    fn stage(&self) -> &'static str {
        todo!()
    }
}

// --- check (do not modify) ---
let err = ingest("abc").unwrap_err();
match err.cause() {
    FeedError::BadQuote { raw } => assert_eq!(raw, "abc"),
    other => panic!("unexpected cause: {other}"),
}
assert_eq!(err.stage(), "ingest");
println!("Exercise 3.2 passed");


### Exercise 4.1 — Into the catch-all

**Goal**: return a `Box<dyn Error>` from glue code, converting leaf errors with nothing but std's blanket `From`.

**Given**: `FeedError` from §2.1 and `parse_qty` from §2.1 are both live; the box accepts any `Error + 'static` type through `From`.

**Objectives**

- implement `boxed_qty` to propagate the parse failure with `?` and return `Ok(qty)` on success — the signature does the conversion
- keep the signature unchanged
- confirm the trap: inside the box, the error's variants are erased — no `match` exists on a `Box<dyn Error>`, only downcasts

**See**: §4.1 of the example notebook.

**Hint**: the body is two lines; the `?` picks up std's `impl<E: Error + 'static> From<E> for Box<dyn Error>` and needs no bridge of yours.


In [ ]:
// (std::error::Error is in scope from 1.2's use line.)

/// Glue: too small to own a domain error — hand the cause to the catch-all.
///
/// TODO: replace `todo!()` with the two-line body.
fn boxed_qty(raw: &str) -> Result<u32, Box<dyn Error>> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(boxed_qty("42").expect("42 parses"), 42);
let bad = boxed_qty("x").unwrap_err();
assert_eq!(bad.to_string(), "malformed quote: \"x\""); // Display through the vtable
assert!(bad.downcast_ref::<FeedError>().is_some()); // the Any machinery, again
let msg: Box<dyn Error> = "bad config".into(); // &str is an Error too
assert_eq!(msg.to_string(), "bad config");
println!("Exercise 4.1 passed");


### Exercise 4.2 — The chain survives erasure

**Goal**: prove the §3 chain walk runs unchanged on boxed errors — `source()` is a trait method, so the vtable serves it without knowing a single concrete type.

**Objectives**

- implement `boxed_chain` to box the given wrapper and collect the same two-headline chain `chain_of` produced
- keep the signature unchanged; reuse the walk shape from §3.1 (first hop fully qualified, then trait-object hops)
- keep the downcast assertion in the check honest: the box must still hand back a `PipelineError`

**See**: §4.2 of the example notebook.

**Hint**: `Box::new(err)` coerces to `Box<dyn Error>` at the `into()`; from there the walk is the §3.1 loop with `e` already a `&dyn Error`.


In [ ]:
/// The §3 chain, run through the catch-all: source() is a trait method.
///
/// TODO: replace `todo!()` with the box-plus-walk body.
fn boxed_chain(err: PipelineError) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let err = PipelineError::Stage {
    source: FeedError::Timeout(250),
    stage: "normalize",
};
let chain = boxed_chain(err);
assert_eq!(chain.len(), 2);
assert_eq!(chain[0], "pipeline failed at stage normalize");
assert_eq!(chain[1], "timeout after 250 ms");
println!("Exercise 4.2 passed");


### Exercise 5.1 — The main contract, acted out

**Goal**: reproduce what a binary's stderr and exit code look like on the `Err` path of `main() -> Result`, using the manual exit for the Display form.

**Given**: the kernel refuses a user-defined `main`, so the contract is acted out about a nested `run` — exactly as §5.1 of the example notebook does.

**Objectives**

- implement `run` to return `Err(PriceError(px))` when `px` is zero or above 10 000, `Ok(())` otherwise — **every** path must end with a `Result` (the E0308 rule)
- implement `report` to return the exact `main`-style stderr line for the `Err` case: `Error: ` followed by the error's **Debug**, and `ok` for the `Ok` case
- keep the signatures unchanged

**See**: §5.1 of the example notebook.

**Hint**: the Debug form is `{e:?}` — quotes only appear when the error type is `String`; a `match` with two arms keeps both paths honest.


In [ ]:
#[derive(Debug, Clone, PartialEq)]
struct PriceError(u32);
/// Validate a price; zero and anything above 10 000 are refused.
///
/// TODO: replace `todo!()` with the two-path body.
fn run(px: u32) -> Result<(), PriceError> {
    todo!()
}
/// The line main() -> Result would put on stderr (or `ok` on the 0-exit path).
///
/// TODO: replace `todo!()` with the match over run(px).
fn report(px: u32) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(run(4_125), Ok(()));
assert!(matches!(run(0), Err(PriceError(0))));
assert!(matches!(run(10_001), Err(PriceError(10_001))));
assert_eq!(report(4_125), "ok");
assert_eq!(report(0), "Error: PriceError(0)"); // Debug - no Display here
println!("Exercise 5.1 passed");


### Exercise 6.1 — The two channels

**Goal**: classify failures correctly, then catch a panic and read its payload type.

**Objectives**

- implement `validate` as the `Result` story: `Err(PriceError(px))` outside the band, `Ok` inside
- implement `must_accept` as the panic story: `panic!("hard stop on {px}")` when the price is outside the band
- implement `payload_kind` to return `"str"` when a caught payload downcasts to `&'static str`, `"string"` when it downcasts to `String`, and `"other"` otherwise
- keep the signatures unchanged

**See**: §6.1 of the example notebook.

**Hint**: `panic::catch_unwind(|| must_accept(0))` returns `Err(Box<dyn Any + Send>)` — `downcast_ref::<&'static str>()` first (literal panics), then `String` (formatted panics).


In [ ]:
// (std::panic is imported here for the rest of the notebook.)
use std::panic;

/// Recoverable: the caller decides.
///
/// TODO: replace `todo!()` with the band check.
fn validate(px: u32) -> Result<(), PriceError> {
    todo!()
}
/// A bug, not a failure: the panic channel.
///
/// TODO: replace `todo!()` with the band check that panics.
fn must_accept(px: u32) {
    todo!()
}
/// What type is the payload really?
///
/// TODO: replace `todo!()` with the downcast ladder.
fn payload_kind(payload: Box<dyn std::any::Any + Send>) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(validate(4_125), Ok(()));
assert!(matches!(validate(0), Err(PriceError(0))));
let caught = panic::catch_unwind(|| must_accept(0));
assert!(caught.is_err());
assert_eq!(payload_kind(caught.unwrap_err()), "string"); // formatted panic
let caught_lit = panic::catch_unwind(|| panic!("hard stop"));
assert_eq!(payload_kind(caught_lit.unwrap_err()), "str"); // literal panic
println!("Exercise 6.1 passed");


### Exercise 6.2 — The hook counts, the catch resumes

**Goal**: install a panic hook that observes (but cannot stop) unwinding, and confirm the ordering: hook first, `catch_unwind` second.

**Given**: a `PANICS` counter and the hook installation are given — the kernel's own hook is saved with `take_hook()` and restored with `set_hook`, so the demo is re-runnable.

**Objectives**

- implement `record` to be the hook's body: increment `PANICS`
- after the given hook runs and one panic is caught, implement `hook_saw` to return the counter's current value
- keep the signatures unchanged; leave the given take/set pair exactly where it is

**See**: §6.2 of the example notebook.

**Hint**: `PANICS.fetch_add(1, Ordering::Relaxed)` is the whole body; the count is read with `PANICS.load(Ordering::Relaxed)` *after* the catch, which proves the hook ran before it.


In [ ]:
// (AtomicUsize/Ordering are imported here for the rest of the notebook.)
use std::sync::atomic::{AtomicUsize, Ordering};
static PANICS: AtomicUsize = AtomicUsize::new(0);
/// The hook's body: observe one panic. Hooks cannot stop the unwind.
///
/// TODO: replace `todo!()` with the increment.
fn record() {
    todo!()
}
/// How many panics has the hook observed so far?
///
/// TODO: replace `todo!()` with the load.
fn hook_saw() -> usize {
    todo!()
}

// --- check (do not modify) ---
let previous = panic::take_hook();
panic::set_hook(Box::new(|_info| record()));
let before = hook_saw();
let _ = panic::catch_unwind(|| panic!("observed, then caught"));
let after = hook_saw();
panic::set_hook(previous);
assert_eq!(after, before + 1); // hook ran, unwind happened, catch resumed
println!("Exercise 6.2 passed");


### Exercise 7.1 — The backtrace field

**Goal**: store a `Backtrace` inside an error at construction time, and read its status — the stable pattern the nightly provider API will one day replace the *access* for.

**Given**: the `SnapshotError` struct and its `Display` are given; only the `Error` impl and the constructor are missing.

**Objectives**

- read the given empty `Error` impl: no cause to expose, so the default `source()` is the whole answer (§1.1's lesson, now routine)
- implement `snap` to build the error with `Backtrace::capture()` in the field
- implement `forced` to build it with `Backtrace::force_capture()`
- keep the signatures unchanged

**See**: §7.1 and §7.2 of the example notebook.

**Hint**: struct literal with `bt: Backtrace::capture()` — and `use std::backtrace::Backtrace;` first; the status assertions in the check cover only what is deterministic (forced is always `Captured`).


In [ ]:
// (Backtrace/BacktraceStatus are imported here for the rest of the notebook.)
use std::backtrace::{Backtrace, BacktraceStatus};

#[derive(Debug)]
struct SnapshotError {
    msg: &'static str,
    bt: Backtrace,
}
impl Display for SnapshotError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "snapshot failed: {}", self.msg)
    }
}
// The empty body is the point: no cause, so the default source() is correct.
impl std::error::Error for SnapshotError {}

/// Capture at construction: gated by RUST_BACKTRACE (see 09e 6.2).
///
/// TODO: replace `todo!()` with the struct literal.
fn snap(msg: &'static str) -> SnapshotError {
    todo!()
}
/// Capture regardless of the gate: always Captured.
///
/// TODO: replace `todo!()` with the struct literal.
fn forced(msg: &'static str) -> SnapshotError {
    todo!()
}

// --- check (do not modify) ---
let a = snap("disk full");
assert_eq!(a.to_string(), "snapshot failed: disk full");
assert!(matches!(
    a.bt.status(),
    BacktraceStatus::Disabled | BacktraceStatus::Captured
));
let b = forced("disk full");
assert!(matches!(b.bt.status(), BacktraceStatus::Captured));
assert!(format!("{b:?}").contains("bt:")); // the field is in the Debug dump
println!("Exercise 7.1 passed");


### Exercise 8.1 — The size table

**Goal**: re-derive the chapter's size facts with `size_of`, and explain the niche through two layers in one assert-laden cell.

**Given**: `PriceError(u32)` from §5.1 and the `Big` enum below are given.

**Objectives**

- implement `size_of_result` to return `size_of::<Result<u32, PriceError>>()`
- implement `niche_two_layers` to return `size_of::<Option<Result<u32, ()>>>()`
- implement `box_error_size` to return `size_of::<Result<u32, Box<dyn std::error::Error>>>()`
- keep the signatures unchanged

**See**: §8.1 of the example notebook.

**Hint**: each body is one `size_of::<..>()` call — the assertions carry the facts; the exercise is knowing which generic arguments to spell.


In [ ]:
enum Big {
    A(u64, u64, u64),
    B(u64, u64, u64),
}
/// Result<u32, PriceError> — the small, register-friendly shape.
///
/// TODO: replace `todo!()` with the size_of call.
fn size_of_result() -> usize {
    todo!()
}
/// Option<Result<u32, ()>> — the niche through two layers.
///
/// TODO: replace `todo!()` with the size_of call.
fn niche_two_layers() -> usize {
    todo!()
}
/// Result<u32, Box<dyn Error>> — the fat-pointer error.
///
/// TODO: replace `todo!()` with the size_of call.
fn box_error_size() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(size_of_result(), 8);
assert_eq!(niche_two_layers(), 8); // None hides inside Result's discriminant space
assert_eq!(box_error_size(), 16); // the box is a fat pointer
let bigs = [Big::A(1, 2, 3), Big::B(4, 5, 6)];
let mut checksum = 0u64;
for b in &bigs {
    match b {
        Big::A(x, y, z) => checksum += x + y + z,
        Big::B(x, y, z) => checksum += x + y + z,
    }
}
assert_eq!(checksum, 21);
assert_eq!(size_of::<Big>(), 32);
assert_eq!(size_of::<Result<Big, ()>>(), 32); // tag hides in padding
println!("Exercise 8.1 passed");


### Exercise 9.1 — The unwrap payload, read on purpose

**Goal**: catch what `unwrap()` really panics with, and state the `unwrap`-vs-`expect` contract in code.

**Objectives**

- implement `unwrap_text` to catch the panic of `r.unwrap()` and return the payload as `&str` (the payload of a formatted panic is a `String`)
- implement `expect_text` likewise for `r.expect("price validated upstream")`
- keep the signatures unchanged

**See**: §9.1 of the example notebook.

**Hint**: `panic::catch_unwind(|| r.unwrap()).unwrap_err()` then `downcast_ref::<String>()`; the two texts share the `PriceError(9)` Debug tail — only the prefix differs.


In [ ]:
/// The full panic text of r.unwrap(), payload downcast to &str.
///
/// TODO: replace `todo!()` with the catch-and-downcast.
fn unwrap_text(r: Result<u32, PriceError>) -> String {
    todo!()
}
/// The full panic text of r.expect("price validated upstream").
///
/// TODO: replace `todo!()` with the catch-and-downcast.
fn expect_text(r: Result<u32, PriceError>) -> String {
    todo!()
}

// --- check (do not modify) ---
let r: Result<u32, PriceError> = Err(PriceError(9));
let u = unwrap_text(r.clone());
assert!(u.contains("called `Result::unwrap()` on an `Err` value"));
assert!(u.contains("PriceError(9)")); // Debug of the error, not its Display
let e = expect_text(r);
assert!(e.starts_with("price validated upstream")); // the call site's intent
assert!(e.contains("PriceError(9)"));
println!("Exercise 9.1 passed");


## 11. Review: Mixed Problems

Four problems, each assembling several sections' worth of machinery: the full three-type chain, the blanket-`From` wall and its legal escape, both channels for one failure, and the wrapper audit that closes the loop back to §1.

### Exercise 11.1 — Assemble the layered story

**Goal**: one function that builds the full cast — parse, handshake, wrap — and returns an error whose chain reads bottom-up through three types.

**Given**: `FeedError`, `PipelineError` and `parse_qty` from earlier sections are live. `SessionError` is declared with one variant `Ingest { source: PipelineError, session: u64 }`, its `Display`, and its `Error` impl with `source()`.

**Objectives**

- implement `From<PipelineError> for SessionError` stamping `session: 7`
- implement `session_ingest` to call `ingest` and propagate its result with `?` — the leaf-to-wrapper hop already happened inside `ingest`; your `From` handles the wrapper-to-session hop
- keep the signatures unchanged

**See**: §2.1, §3.1 and §10 of the example notebook.

**Hint**: two `From` impls now chain: `FeedError → PipelineError` (§2.1's, used inside `ingest`) then `PipelineError → SessionError` (yours); the `?` composes them without naming either conversion at the callsite.


In [ ]:
#[derive(Debug)]
enum SessionError {
    Ingest { source: PipelineError, session: u64 },
}
impl Display for SessionError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            SessionError::Ingest { source, session } => {
                write!(f, "session {session}: {source}")
            }
        }
    }
}
impl std::error::Error for SessionError {
    fn source(&self) -> Option<&(dyn std::error::Error + 'static)> {
        match self {
            SessionError::Ingest { source, .. } => Some(source),
        }
    }
}
impl From<PipelineError> for SessionError {
    /// The second bridge: the session layer wraps the pipeline's error.
    ///
    /// TODO: replace `todo!()` with the Ingest variant, session stamped 7.
    fn from(source: PipelineError) -> Self {
        todo!()
    }
}

/// Top layer: the signature promises SessionError; one ? does the work.
///
/// TODO: replace `todo!()` with the body: ingest(raw)?, then Ok(qty).
fn session_ingest(raw: &str) -> Result<u32, SessionError> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(session_ingest("300").expect("300 parses"), 300);
let err = session_ingest("abc").unwrap_err();
assert_eq!(err.to_string(), "session 7: pipeline failed at stage ingest");
assert!(err.source().is_some());
println!("Exercise 11.1 passed");


### Exercise 11.2 — Two-character bridge: the blanket `From` trap

**Goal**: feel the E0119 wall the example notebook catalogued — the blanket "accept anything" impl that cannot exist, and the wrapper that achieves its goal legally.

**Objectives**

- read the two commented impls: the blanket `impl<E: std::error::Error + 'static> From<E> for PriceError` is **E0119** — it collides with std's reflexive `impl<T> From<T> for T`
- implement `string_error` to build a `Box<dyn Error>` from a `&'static str` message — the legal way to accept "anything stringy"
- keep the signatures unchanged

**See**: §9 of the example notebook.

**Hint**: the body is one `.into()`; the escape from E0119 is a *wrapper* type with the blanket impl, which is exactly what `Box<dyn Error>` already is.


In [ ]:
// The tempting impl, and the wall it hits (uncomment to see E0119):
//
// impl<E: std::error::Error + 'static> From<E> for PriceError {
//     fn from(_e: E) -> Self { PriceError(0) }
// }
//
// error[E0119]: conflicting implementations of trait `From<PriceError>` for
//               type `PriceError`
//   = note: conflicting implementation in crate `core`:
//           - impl<T> From<T> for T;

/// The legal catch-all for messages: std's From<str> impls do the work.
///
/// TODO: replace `todo!()` with the one-expression body.
fn string_error(msg: &'static str) -> Box<dyn Error> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(string_error("bad config").to_string(), "bad config");
assert!(string_error("feed down").source().is_none()); // no cause: chain ends
println!("Exercise 11.2 passed");


### Exercise 11.3 — Recoverable vs bug, decided twice

**Goal**: take one failure and implement it on both channels, so the choice between them is yours to make twice.

**Given**: the price band of §6.1 (refuse zero and anything above 10 000).

**Objectives**

- implement `checked_total` to sum the prices with the `Result` story: the first out-of-band price becomes `Err(PriceError(px))`
- implement `total_or_die` to run the same sum but `panic!("bad price {px}")` on the first out-of-band price
- implement `total_catching` to run `total_or_die` under `catch_unwind` and return the checksum as `Result<u64, ()>` — `None`-style: the panic became an `Err`, the process lives
- keep the signatures unchanged

**See**: §6.1 and §6.2 of the example notebook.

**Hint**: `prices.iter().try_fold(0u64, |acc, p| ...)` is the natural `Result` shape; the catching wrapper is `match panic::catch_unwind(|| total_or_die(prices)) { Ok(t) => Ok(t), Err(_) => Err(()) }`.


In [ ]:
/// The Result story: first bad price aborts the sum with a typed error.
///
/// TODO: replace `todo!()` with the try_fold body.
fn checked_total(prices: &[u32]) -> Result<u64, PriceError> {
    todo!()
}
/// The panic story: same rule, different channel.
///
/// TODO: replace `todo!()` with the loop that panics.
fn total_or_die(prices: &[u32]) -> u64 {
    todo!()
}
/// The panic, caught: the checksum as a Result, the process still alive.
///
/// TODO: replace `todo!()` with catch_unwind plus the match.
fn total_catching(prices: &[u32]) -> Result<u64, ()> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(checked_total(&[4_125, 4_126]).expect("in band"), 8_251);
assert!(matches!(checked_total(&[4_125, 0]), Err(PriceError(0))));
assert_eq!(total_or_die(&[4_125, 4_126]), 8_251);
assert_eq!(total_catching(&[4_125, 4_126]), Ok(8_251));
assert_eq!(total_catching(&[4_125, 0]), Err(()));
println!("Exercise 11.3 passed");


### Exercise 11.4 — The wrapper audit

**Goal**: return to §1's wrapper with everything the chapter taught since: chain, source, downcast, size — one audit function that answers all four questions about any `DeskError`.

**Given**: `RiskError` and `DeskError` from §1 — `DeskError` wraps a `String`-carrying variant plus a `&'static str`, so it is a reference for the "does my wrapper fit in registers?" question.

**Objectives**

- implement `audit_source` to return the chain length under the given error (1 = no cause, 2 = one cause, …)
- implement `audit_size` to return `size_of::<DeskError>()`
- keep the signatures unchanged
- the check's commented block is the answer key for the design questions: which accessors survive erasure, and what the box costs

**See**: §3.1, §4.1 and §8.1 of the example notebook.

**Hint**: reuse the §3.1 walk; `size_of` needs no instance — the type alone decides.


In [ ]:
/// How long is the cause chain under this error?
///
/// TODO: replace `todo!()` with the chain walk, returning the count.
fn audit_source(err: &dyn std::error::Error) -> usize {
    todo!()
}
/// What does the wrapper cost, inline?
///
/// TODO: replace `todo!()` with the size_of call.
fn audit_size() -> usize {
    todo!()
}

// --- check (do not modify) ---
let plain = RiskError::Halted("circuit breaker");
assert_eq!(audit_source(&plain), 1); // Error with no source: chain of one
let wrapped = DeskError::Feed {
    source: RiskError::LimitBreach { symbol: "ES".to_string(), px: 4_125 },
    desk: "rates",
};
assert_eq!(audit_source(&wrapped), 2);
assert!(audit_size() >= 24); // &'static str + String + discriminant, padded
// Boxed, the chain still walks but the match does not:
let boxed: Box<dyn Error> = Box::new(wrapped);
assert_eq!(audit_source(&*boxed), 2);
// boxed.downcast_ref::<DeskError>() is the only way back to the variants.
println!("Exercise 11.4 passed");


### What Comes Next

Chapter 11's error types are now yours to write without looking anything up. The track continues:

| Chapter | Title | What it settles |
|---|---|---|
| **12** | **Iterators in Depth** | the `Iterator` trait from the author's side — `try_fold`, the loop `?` wishes it were |
| 13 | Modules, Crates and Testing | `pub` finally bites, and `#[should_panic]` tests the §6 channel |
| 14 | Smart Pointers and Interior Mutability | `Box` from §4 gets siblings |

**Carry forward.** The six moves this notebook drilled: the three-impl error enum, the `From` bridge and its `map_err` escape, the `source()` walk, the catch-all and its downcasts, the `main() -> Result` contract, and the panic/`Result` boundary.
